# Long-lived eddies: depth-following PV-gradient case studies

This is the depth-resolved counterpart of `long_eddies.ipynb`. It loads the cached depth-following tables rather than recalculating ellipse means. Each colour represents the same target depth in every time-series panel and on the map. Values are interpolated only between adjacent valid fitted levels; no vertical extrapolation is allowed. Blue and orange background shading use the depth-averaged snapshot classification so each time has one unambiguous regime.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

HERE = Path.cwd().resolve()
ANALYSIS_ROOT = next((p for p in (HERE, *HERE.parents) if (p / 'seacofs_tilt_tools.py').exists()), None)
if ANALYSIS_ROOT is None:
    raise FileNotFoundError('Run from seacofs_eddy_tilt_analysis or one of its subfolders.')
if str(ANALYSIS_ROOT) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_ROOT))
import seacofs_tilt_tools as tilt
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 80)

## Settings and cached data

In [ ]:
TARGET_DEPTHS_M = (50.0, 200.0, 500.0, 750.0, 1000.0)
MAX_INTERPOLATION_GAP_M = 200.0
DOMINANCE_FACTOR = 2.0
N_PER_POLARITY = 6

paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
snapshot_df = tilt.add_pv_gradient_terms(source='depth_snapshot')
depth_df = tilt.add_pv_gradient_terms(source='depth')

assert not snapshot_df.duplicated(['Eddy', 'Day']).any()
assert not depth_df.duplicated(['Eddy', 'Day', 'Depth']).any()
print(f'{len(snapshot_df):,} snapshots; {len(depth_df):,} depth rows')

## Select the longest-lived eddies

Selection uses one row per eddy-day. If `Age` is unavailable, observed day span is used.

In [ ]:
if 'Age' in snapshot_df:
    lifetime = snapshot_df.groupby(['Cyc', 'Eddy']).Age.first()
else:
    lifetime = snapshot_df.groupby(['Cyc', 'Eddy']).Day.agg(np.ptp)
selected_ids = (lifetime.groupby(level='Cyc').nlargest(N_PER_POLARITY)
                .index.get_level_values('Eddy').unique())
selected = snapshot_df[snapshot_df.Eddy.isin(selected_ids)].copy()
display(lifetime.loc[lifetime.index.get_level_values('Eddy').isin(selected_ids)]
        .rename('lifetime_days').reset_index().sort_values(['Cyc', 'lifetime_days'], ascending=[True, False]))

## Interpolate depth profiles without extrapolation

PV-gradient magnitudes, directions and dominance are reconstructed after interpolation of east/north vector components. They are never interpolated directly.

In [ ]:
PROFILE_COLUMNS = [
    'xc', 'yc', 'h', 'PV', 'Ro', 'beta', 'abs_vort',
    'PV_grad_plan_x', 'PV_grad_plan_y',
    'PV_grad_topo_x', 'PV_grad_topo_y',
]

def interpolate_value(depth, values, target, max_gap_m=MAX_INTERPOLATION_GAP_M):
    depth = np.asarray(depth, float)
    values = np.asarray(values, float)
    valid = np.isfinite(depth) & np.isfinite(values)
    depth, values = depth[valid], values[valid]
    if not len(depth):
        return np.nan
    order = np.argsort(depth)
    depth, values = depth[order], values[order]
    exact = np.flatnonzero(np.isclose(depth, target, atol=1e-6, rtol=0))
    if len(exact):
        return values[exact[0]]
    upper = int(np.searchsorted(depth, target))
    lower = upper - 1
    if lower < 0 or upper >= len(depth):
        return np.nan
    if depth[upper] - depth[lower] > max_gap_m:
        return np.nan
    weight = (target - depth[lower]) / (depth[upper] - depth[lower])
    return (1-weight)*values[lower] + weight*values[upper]

def fixed_depth_track(eddy, target_depths=TARGET_DEPTHS_M):
    source = depth_df[depth_df.Eddy.eq(eddy)].sort_values(['Day', 'Depth'])
    rows = []
    for day, profile in source.groupby('Day', sort=True):
        z = profile.Depth.to_numpy(float)
        for target in target_depths:
            row = {'Eddy': eddy, 'Day': day, 'TargetDepth': target}
            for column in PROFILE_COLUMNS:
                row[column] = interpolate_value(z, profile[column], target)
            rows.append(row)
    out = pd.DataFrame(rows)
    out['PV_grad_x'] = out.PV_grad_plan_x + out.PV_grad_topo_x
    out['PV_grad_y'] = out.PV_grad_plan_y + out.PV_grad_topo_y
    for prefix in ('PV_grad_plan', 'PV_grad_topo', 'PV_grad'):
        out[f'{prefix}_mag'] = np.hypot(out[f'{prefix}_x'], out[f'{prefix}_y'])
        out[f'{prefix}_theta'] = tilt.bearing_from_xy(out[f'{prefix}_x'], out[f'{prefix}_y'])
    with np.errstate(divide='ignore', invalid='ignore'):
        out['topo_plan_ratio'] = np.log(out.PV_grad_topo_mag/out.PV_grad_plan_mag)
    return out

## Depth-resolved time plot

The original tilt/shear/ellipse/delta-theta additions are deliberately omitted here. The map uses depth colour rather than eddy age because depth is the comparison of interest.

In [ ]:
def time_plot_depth(eddy, snapshot_data=snapshot_df, grid=grid):
    snap = snapshot_data[snapshot_data.Eddy.eq(eddy)].sort_values('Day').copy()
    if snap.empty:
        raise ValueError(f'Eddy {eddy} is not present in snapshot_data')
    prof = fixed_depth_track(eddy)
    t0 = snap.Day.iloc[0]
    snap['t'] = snap.Day - t0
    prof['t'] = prof.Day - t0
    cyc = snap.Cyc.iloc[0]

    depth_colors = dict(zip(TARGET_DEPTHS_M, plt.cm.viridis(np.linspace(.08, .92, len(TARGET_DEPTHS_M)))))
    PLAN_COLOR, TOPO_COLOR = 'tab:blue', 'tab:orange'
    threshold = np.log(DOMINANCE_FACTOR)
    planetary = snap.topo_plan_ratio_smooth <= -threshold
    topographic = snap.topo_plan_ratio_smooth >= threshold

    fig = plt.figure(figsize=(13, 10))
    gs = fig.add_gridspec(5, 2, width_ratios=[2.2, 1])
    axs = [fig.add_subplot(gs[i, 0]) for i in range(5)]
    axm = fig.add_subplot(gs[:, 1])
    for ax in axs:
        for t in snap.loc[planetary, 't']:
            ax.axvspan(t-.5, t+.5, color=PLAN_COLOR, alpha=.2, lw=0)
        for t in snap.loc[topographic, 't']:
            ax.axvspan(t-.5, t+.5, color=TOPO_COLOR, alpha=.2, lw=0)

    axs[0].plot(snap.t, snap.TiltDis, color='tab:purple', lw=1.7)
    axs[0].set_ylabel('Tilt distance [km]', color='tab:purple')
    axs[0].tick_params(axis='y', labelcolor='tab:purple')
    ax0 = axs[0].twinx()
    for depth, part in prof.groupby('TargetDepth'):
        ax0.plot(part.t, part.PV_grad_mag, color=depth_colors[depth], lw=1.2, label=f'{depth:g} m')
    ax0.set_ylabel(r'$|\nabla PV|$', color='tab:green')
    ax0.tick_params(axis='y', labelcolor='tab:green')
    ax0.legend(ncol=3, fontsize=7, frameon=False)

    for depth, part in prof.groupby('TargetDepth'):
        axs[1].plot(part.t, part.topo_plan_ratio, color=depth_colors[depth], lw=1.3, label=f'{depth:g} m')
    axs[1].axhline(0, color='0.2', lw=.8)
    axs[1].axhline(threshold, color=TOPO_COLOR, ls='--', lw=.8)
    axs[1].axhline(-threshold, color=PLAN_COLOR, ls='--', lw=.8)
    axs[1].set_ylabel(r'$\ln(|\nabla PV|_{topo}/|\nabla PV|_{plan})$')

    for depth, part in prof.groupby('TargetDepth'):
        axs[2].plot(part.t, part.Ro, color=depth_colors[depth], lw=1.3)
    axs[2].axhline(1, color='0.3', ls=':', lw=.8)
    axs[2].set_ylabel(r'$Ro$')

    ax3 = axs[3].twinx()
    for depth, part in prof.groupby('TargetDepth'):
        color = depth_colors[depth]
        axs[3].plot(part.t, part.PV, color=color, lw=1.3)
        ax3.plot(part.t, part.h/1e3, color=color, lw=1.0, ls='--', alpha=.75)
    axs[3].set_ylabel('PV (solid)')
    ax3.set_ylabel('Depth [km] (dashed)')
    ax3.invert_yaxis()

    ax4 = axs[4].twinx()
    for depth, part in prof.groupby('TargetDepth'):
        color = depth_colors[depth]
        axs[4].plot(part.t, part.beta, color=color, lw=1.3)
        ax4.plot(part.t, part.abs_vort, color=color, lw=1.0, ls='--', alpha=.75)
    axs[4].set(xlabel='Eddy age [days]', ylabel=r'$\beta$ (solid)')
    ax4.set_ylabel(r'$\zeta+f$ (dashed)')

    for ax in axs:
        ax.grid(alpha=.2)
        ax.margins(x=0)

    map_rows = prof.dropna(subset=['xc', 'yc'])
    pad = 20
    xmin, xmax = map_rows.xc.min()-pad, map_rows.xc.max()+pad
    ymin, ymax = map_rows.yc.min()-pad, map_rows.yc.max()+pad
    inside = ((grid.X_grid >= xmin) & (grid.X_grid <= xmax) &
              (grid.Y_grid >= ymin) & (grid.Y_grid <= ymax))
    bathy = np.where((grid.mask_rho == 1) & inside, grid.h/1e3, np.nan)
    cf = axm.contourf(grid.X_grid, grid.Y_grid, bathy, cmap='Greys_r')
    fig.colorbar(cf, ax=axm, orientation='horizontal', location='bottom',
                 label='Depth [km]', shrink=.8, pad=.08)
    for _, day_part in map_rows.groupby('Day'):
        day_part = day_part.sort_values('TargetDepth')
        axm.plot(day_part.xc, day_part.yc, color='0.4', alpha=.12, lw=.7)
    for depth, part in map_rows.groupby('TargetDepth'):
        axm.plot(part.xc, part.yc, '.-', color=depth_colors[depth],
                 lw=1.5, ms=3, label=f'{depth:g} m')
    axm.set(xlim=(xmin, xmax), ylim=(ymin, ymax), xlabel='x [km]', ylabel='y [km]')
    axm.set_aspect('equal')
    axm.legend(title='Target depth', ncol=2, fontsize=7, frameon=False)
    fig.suptitle(f'{cyc}{eddy}: depth-following PV gradients', y=.995)
    plt.tight_layout()
    plt.show()
    return fig, axs, axm

## Generate the selected cases

In [ ]:
for eddy in selected.Eddy.unique():
    time_plot_depth(eddy, selected, grid)